## DSPy Unsloth-Llama-3-8B-4bit Information Extraction

#### Install Library Dependies (Run on Google Colab)

In [1]:
%%capture
%pip install dspy-ai -U transformers chromadb accelerate sentence-transformers bitsandbytes peft rich

#### Load in Python Libraries

In [2]:
import dspy
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

#### Set LLM Llama 3 post-quantized 4 bit model from Unsloth

In [ ]:
llm = dspy.HFModel(model="unsloth/llama-3-8b-Instruct-bnb-4bit", hf_device_map='auto', model_kwargs= {'temperature':0.0,'do_sample': False})
dspy.settings.configure(lm = llm)

#### Create Model Signature and Module

In [8]:
# create signature showing input and output

class GenerateAnswer(dspy.Signature):
    """Answer questions with short factoid answers."""

    context = dspy.InputField(desc="contain relevant facts")
    question = dspy.InputField(desc="unique possible questions")
    answer = dspy.OutputField(desc="key-value pairs, each answer must be between 1 and 20 words")



# Create module using dspy module
class QUESTIONANSWER(dspy.Module):
    def __init__(self,question):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer, max_tokens=400)
        self.question=question

    def forward(self, context):
        question=self.question
        pred = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context,answer=pred.answer)

#### Test Example 

In [9]:
example1 = """
6 hours\nAssociate Actuary - SPA Rx\nCincinnati, OH 45217\n**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting bids, filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of situations or data requires an in-depth evaluation of variable factors. **Responsibilities** _This a remote nationwide position_ The Associate Actuary, Pricing establishes market level financial metrics to align with segment profitability goals, analyzes market level results and projections and develops recommended pricing actions to address gaps to targeted metrics. Leverages market level projections and experience data tools to research root cause and capture insights. Researches and understands competitors in marketplace and collaborates with sales and other partners supporting the markets to develop strategies for profitable membership growth. Accountable for actuarial certifications on rate filings, including attesting to compliance with state and federal rating and benefit regulations. Begins to influence department's strategy. Makes decisions on moderately complex to complex issues regarding technical approach for project components, and work is performed without direction. Exercises considerable latitude in determining objectives and approaches to assignments. **Required Qualifications** + Bachelor's Degree + Associate of Society of Actuaries (ASA) designation + Meets eligibility requirements for Humana's Actuarial Professional Development Program (APDP) + MAAA + Strong communication + Must be passionate about contributing to an organization focused on continuously improving consumer experiences **Our Hiring Process** As part of our hiring process for this opportunity, we may contact you via text message and email to gather more information using a software platform called Modern Hire. Modern Hire Text, Scheduling and Video technologies allow you to interact with us at the time and location most convenient for you. If you are selected to move forward from your application prescreen, you may receive correspondence inviting you to participate in a pre-recorded Voice, Text Messaging and/or Video interview. Your recorded interview will be reviewed and you will subsequently be informed if you will be moving forward to next round of interviews If you have additional questions regarding this role posting and are an Internal Candidate, please send them to the Ask A Recruiter persona by visiting go/Buzz and searching Ask A Recruiter! Please be sure to provide the requisition number so we may be able to research your request quicker. **Alert:** Humana values personal identity protection. Please be aware that applicants selected for leader review may be asked to provide a social security number, if it is not already on file. When required, an email will be sent from Humana@myworkday.com with instructions to add the information into the application at Humana's secure website. **_Humana is more than an equal opportunity employer, Humana's dedication to promoting diversity, multiculturalism, and inclusion is at the heart of what we do in all of our Humana roles. Diversity is more than a commitment to us, it is the foundation of what we do. We are fully focused on diversity of race, gender, sexual orientation, religion, ethnicity, national origin and all of the other fascinating characteristics that make us each uniquely wonderful._** \#LI-Remote **Scheduled Weekly Hours** 40 Humana complies with all applicable federal civil rights laws and does not discriminate on the basis of race, color, national origin, age, disability, sex, sexual orientation, gender identity or religion. We also provide free language interpreter services. See our https://www.humana.com/legal/accessibility-resources?source=Humana_Website.
"""

uncompiled_fs=QUESTIONANSWER('''
                             1. What is the title of this position?
                             2. Where is this position located, including city, state and zip code?
                             3. What is the work arrangement for this position, remote, hybrid, or on-site?
                             4. what are years of experience required for this position?
                             5. What is the employment type, full time, part time, or internship?
                             6. What is the pay for this position?
                             7. What is required degree or certification?
                             8. What are required skills?
                              ''')

pred = uncompiled_fs(context=example1)
print(pred.answer)
print(llm.inspect_history(n=1))

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


key-value pairs, each answer must be between 1 and 20 words

---

Context:

6 hours
Associate Actuary - SPA Rx
Cincinnati, OH 45217
**Description** The Associate Actuary, Pricing is responsible for setting pricing assumptions, submitting bids, filing and gaining approval of premium rates and rate certifications with regulatory agencies. Supports implementation of rates, new plans and benefit changes. Provides guidance to Product Development on new product/benefit cost impacts. Develops and maintains pricing tools that price standard and custom benefits. The Associate Actuary, Pricing work assignments involve moderately complex to complex issues where the analysis of situations or data requires an in-depth evaluation of variable factors. **Responsibilities** _This a remote nationwide position_ The Associate Actuary, Pricing establishes market level financial metrics to align with segment profitability goals, analyzes market level results and projections and develops recommended pricing 

#### Test on SetFit Filtered Data

In [14]:
data=pd.read_csv('/content/sample_data/AfterProcess_setfit_filter_data.csv')
data.head()

,Unnamed: 0,id,text,word_count_x,word_count_y,body
0,0,0002adbfe84f622360521acd5607f55ab1f07f0a,"['Catholic Health Systems', '3.3', 'Unit Secre...",324,786,"['Catholic Health Systems', '3.3', 'Unit Secre..."
1,1,0004dbdf43f59df618770fbff95aaf81fd90c44f,Job Description: Description IntroductionDo yo...,501,782,Job Description: Description IntroductionDo yo...
2,2,0009cb7c9b52b06ecf6a6a2858b03f3dcb3251b3,Student Assistant-TCHATT/CPAN Texas Tech Univ ...,385,452,Student Assistant-TCHATT/CPAN Texas Tech Univ ...
3,3,000c4b77a7c648fbf1586224608476bde0f97acb,Direct Support Professional ID Home & Communit...,435,655,Direct Support Professional ID Home & Communit...
4,4,000c8f61e1b571dbc0ed0309d392228bbcc41531,un provide jump starts to stranded motorist - ...,393,648,un provide jump starts to stranded motorist - ...


In [15]:
# Run a test on one example (row 1)
example1=data.head(1)['text'].values[0]

def count_words(s):
    words = s.split()
    return len(words)

count_words(example1)

324

In [16]:
uncompiled_fs=QUESTIONANSWER('''
                             1. What is the title of this position?
                             2. Where is this position located, including city, state and zip code?
                             3. What is the work arrangement for this position, remote, hybrid, or on-site?
                             4. what are years of experience required for this position?
                             5. What is the employment type, full time, part time, or internship?
                             6. What is the pay for this position?
                             7. What is required degree or certification?
                             8. What are required skills?
                              ''')

pred = uncompiled_fs(context=example1)
print(pred.answer)
print('-------')
print(llm.inspect_history(n=1))

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


key-value pairs, each answer must be between 1 and 20 words

---

Context: ['Catholic Health Systems', '3.3', 'Unit Secretary Emergency Department MSMH', 'Lewiston, NY', 'Apply on employer site', 'Apply now']['Salary: ', '18.31-23.33 USD', 'Facility:', " Mount St. Mary's Hospital", 'Shift: Shift 1', 'Status:','Full Time ', 'FTE: 1.000000', 'Bargaining Unit:','ACE Associates', 'Exempt from Overtime:','Exempt: No', 'Work Schedule: ', 'Days and Evenings with Weekend and Holiday Rotation', 'Hours: ', '7:30am - 3:30pm', '\nSummary:', '\nUnder the supervision of the Emergency Department Director or designee, the Emergency Department Unit Secretary is responsible for performing general receptionist and clerical duties in the Emergency Department. Maintains established departmental policies and procedures, objectives, quality assurance programs, safety, environmental and infection control standards. Maintains confidentiality of all hospital/staff information.', '\nResponsibilities:', 'EDUCATIO

In [28]:
text=pred.answer

# Extract information after "Answer:"
def answer(text):
  answer_index = text.find("Answer:")
  if answer_index != -1:
      answer_text = text[answer_index + len("Answer:"):].strip()
      answer_items = answer_text.split(", ")
      for item in answer_items:
          print(item)
  else:
      print("The word 'Answer:' was not found in the text.")

answer(text)

1. Unit Secretary Emergency Department MSMH
2. Lewiston
NY
3. On-site
4. Six (6) months to one (1) year
5. Full Time
6. $18.31 - $23.33
7. High School or equivalent
8. Good human relations and oral/written communication skills
Answer telephones
Compile statistics
Develop office procedures
Establish filing systems
Input data into computer programs
Maintain logs
Maintain filing systems
Maintain patient charts
Schedule appointments
Clinical computer information systems
Fax
Photocopier
Stamper
Typewriter.
